# Self-guided lesson — Regular Expressions (Regex) in Python

A **regular expression** (regex) is a pattern used to search, extract, split, or replace text.

Regex is especially useful when the text we want to process follows a **structure** rather than containing one exact value.

Typical data-analysis use cases include:

- extracting IDs or codes from messy text;
- finding dates, phone numbers, or email-like strings;
- removing repeated whitespace;
- standardizing separators;
- splitting text using more than one possible delimiter;
- validating whether strings approximately follow an expected pattern.

The goal of this lesson is not to memorize every regex symbol. Focus on the workflow:

**understand the text → define the pattern → test it → inspect the matches → use it**

## 1. Python's `re` module

Python provides the built-in `re` module for working with regular expressions.

In [1]:
import re

## 2. Use raw strings for regex patterns

Regex uses backslashes for special sequences such as:

- `\d` → digit
- `\w` → word character
- `\s` → whitespace

Python strings also use backslashes for escape sequences. To avoid unnecessary escaping, regex patterns are normally written as **raw strings**:

```python
r"\d+"
```

instead of:

```python
"\\d+"
```

Both can work, but raw strings are usually easier to read.

In [2]:
pattern = r"\d+"
text = "Order 104 contains 3 items."

re.findall(pattern, text)

['104', '3']

## 3. Common building blocks

Some useful regex elements are:

| Pattern | Meaning |
|---|---|
| `\d` | digit |
| `\w` | word character |
| `\s` | whitespace |
| `.` | any character except newline |
| `[abc]` | one character from the set |
| `[^abc]` | one character not in the set |
| `a|b` | `a` or `b` |
| `^` | start of string |
| `$` | end of string |
| `+` | one or more |
| `*` | zero or more |
| `?` | zero or one |
| `{n}` | exactly `n` repetitions |
| `{n,m}` | between `n` and `m` repetitions |

A regex is built by combining small pieces.

For example:

```python
r"\d{3}-\d{2}-\d{4}"
```

means:

- 3 digits
- a hyphen
- 2 digits
- a hyphen
- 4 digits

In [3]:
pattern = r"\d{3}-\d{2}-\d{4}"
text = "Example identifier: 123-45-6789"

re.search(pattern, text)

<re.Match object; span=(20, 31), match='123-45-6789'>

## 4. `re.search()` — find the first match

`re.search(pattern, string)` looks through the string and returns the **first match object** it finds.

If there is no match, it returns `None`.

In [4]:
text = "I have 10 apples and 2 bananas."
pattern = r"\d+"

match = re.search(pattern, text)

match

<re.Match object; span=(7, 9), match='10'>

In [5]:
match.group()

'10'

A match object contains more than the matched text. It also knows **where** the match occurred.

In [6]:
match.start(), match.end(), match.span()

(7, 9, (7, 9))

This is useful when we need the position of the pattern, not only its value.

## 5. `re.match()` vs `re.search()`

`re.match()` checks only **from the beginning of the string**.

`re.search()` searches anywhere in the string.

In [7]:
text = "   Order 123 is ready"
pattern = r"\w+"

print("match :", re.match(pattern, text))
print("search:", re.search(pattern, text))

match : None
search: <re.Match object; span=(3, 8), match='Order'>


Because the string starts with spaces, `re.match()` finds nothing. `re.search()` keeps looking and finds the first word.

For most data-cleaning tasks, `re.search()` is more commonly useful.

## 6. `re.findall()` — extract all matches

When we want every occurrence of a pattern, `re.findall()` is convenient because it returns the matches directly as a list.

In [8]:
text = "I have 3 apples, 5 bananas and 12 oranges."
pattern = r"\d+"

re.findall(pattern, text)

['3', '5', '12']

### Example: alternatives and character classes

In [9]:
text = "The basket contains an apple, a pear and a banana."

fruit_pattern = r"apple|banana|pear"

re.findall(fruit_pattern, text)

['apple', 'pear', 'banana']

In [10]:
text = "Data Science"

vowel_pattern = r"[aeiou]"

re.findall(vowel_pattern, text, flags=re.IGNORECASE)

['a', 'a', 'i', 'e', 'e']

`flags=re.IGNORECASE` allows us to ignore upper/lower case differences without modifying the original text.

## 7. Grouping parts of a pattern

Parentheses `()` create **groups**.

Groups are useful when a match contains several pieces that we want to extract separately.

In [11]:
text = "Invoice INV-2026-0042"
pattern = r"(INV)-(\d{4})-(\d{4})"

match = re.search(pattern, text)

match.groups()

('INV', '2026', '0042')

The complete match is still available with `.group()`, while `.groups()` returns the captured pieces.

In [12]:
match.group(), match.groups()

('INV-2026-0042', ('INV', '2026', '0042'))

## 8. `re.finditer()` — all matches plus positions

`re.finditer()` returns an iterator of match objects.

Use it when you need information such as the position of every match.

In [13]:
text = "Product IDs: 104, 208, 315"
pattern = r"\d+"

for match in re.finditer(pattern, text):
    print(match.group(), match.span())

104 (13, 16)
208 (18, 21)
315 (23, 26)


If you only need the matched values, `findall()` is simpler. If you need metadata about each match, `finditer()` is more useful.

## 9. `re.sub()` — replace patterns

`re.sub(pattern, replacement, string)` replaces every match of the pattern.

In [14]:
text = "Customer   name     contains   irregular spaces."

clean_text = re.sub(r"\s+", " ", text)

clean_text

'Customer name contains irregular spaces.'

This is a common cleaning pattern:

```python
r"\s+"
```

means **one or more whitespace characters**, so any run of spaces/tabs/newlines is replaced by one space.

In [15]:
text = "Order 123 costs 45 euros."

re.sub(r"\d+", "<number>", text)

'Order <number> costs <number> euros.'

## 10. `re.split()` — split using a pattern

Normal `.split()` works well when there is one exact separator.

Regex becomes useful when several separators are possible.

In [16]:
text = "apple,banana;orange | pear"

re.split(r"[,;|]\s*", text)

['apple', 'banana', 'orange ', 'pear']

The pattern means:

- split on comma, semicolon, or `|`
- optionally consume the spaces that follow

## 11. A note about email patterns

Regex is often used to **extract email-like strings**, but validating every valid email address according to all standards is surprisingly complicated.

For data cleaning, a practical pattern can still be useful as long as we understand that it is an approximation.

In [17]:
text = """
Contact john.doe@example.com
or mary_smith@company.co.uk.
Ignore: not-an-email@
"""

email_like_pattern = r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}"

re.findall(email_like_pattern, text)

['john.doe@example.com', 'mary_smith@company.co.uk']

This pattern is appropriate for finding common email-like strings in messy text.

It should **not** be presented as a complete validator for every possible legal email address.

### A useful anchor: `\b` — word boundary

`\b` means **word boundary**.

It does **not** match a visible character. Instead, it matches a **position** where a word character (`\w`) meets a non-word character, or the beginning/end of the string.

For example:

```python
r"\bcat\b"
```

matches `cat` as a complete word:

```text
cat
the cat sleeps
```

but it does not match the `cat` inside:

```text
catalog
scatter
```

This is useful when we want to find a code or word as a separate unit instead of matching the same letters inside a longer string.

## 12. Applied example — cleaning messy product codes

Imagine a column containing product references entered in inconsistent forms:

```text
prd-001
PRD 002
prd_003
Product: PRD-004
```

Suppose the standardized format should be:

```text
PRD-001
```

We can use regex to extract the numeric part and rebuild the code.

In [18]:
raw_codes = [
    "prd-001",
    "PRD 002",
    "prd_003",
    "Product: PRD-004",
    "invalid"
]

In [19]:
def standardize_product_code(value):
    match = re.search(r"(?i)\bprd[-_\s]?(\d{3})\b", value)

    if match:
        return f"PRD-{match.group(1)}"

    return None


[standardize_product_code(value) for value in raw_codes]

['PRD-001', 'PRD-002', 'PRD-003', 'PRD-004', None]

Let's break the pattern down:

```python
r"(?i)\bprd[-_\s]?(\d{3})\b"
```

- `(?i)` → case-insensitive matching
- `\b` → word boundary: require `prd` / the digits to begin or end as separate word units
- `prd` → literal text
- `[-_\s]?` → optional hyphen, underscore, or whitespace
- `(\d{3})` → capture exactly three digits
- `\b` → word boundary: require `prd` / the digits to begin or end as separate word units

The important idea is not the syntax itself. It is that we first identify the **structure** of the values we want to clean.

## 13. Regex or normal string methods?

Do not use regex automatically.

If you need an exact operation, normal string methods are usually easier to read:

```python
text.replace("apple", "orange")
text.split(",")
text.startswith("INV")
```

Regex is most useful when the rule is based on a **pattern**:

```text
one or more digits
several possible separators
optional spaces
a code with a fixed structure
```

Prefer the simplest tool that solves the problem clearly.

# 💡 Check for understanding

A dataset contains the following messy contact strings:

In [20]:
contacts = [
    "Phone: +34 612 345 678",
    "call 612-345-679",
    "Mobile 612345680",
    "no phone provided",
    "Tel: +34-612-345-681"
]

Complete the following tasks:

1. Use `re.findall()` or `re.search()` to identify strings containing a Spanish-style 9-digit mobile number beginning with `6`.
2. Modify the pattern so it works when digits may be separated by spaces or hyphens.
3. Extract the **nine digits only**.
4. Standardize valid numbers to this format:

```text
612 345 678
```

5. Return `None` when no valid number is found.
6. Test your solution against every value in `contacts`.

### Optional challenge

Write a function:

```python
clean_phone(value)
```

that performs the complete process.

After writing it, validate the output manually. Regex can match the pattern you describe, but it cannot tell you whether you described the correct business rule.

# Summary

The most useful regex workflow is:

1. **Understand the structure** of the text.
2. Write a small pattern using raw strings: `r"..."`.
3. Test the pattern on representative examples.
4. Inspect both matches and non-matches.
5. Use:
   - `re.search()` for the first match;
   - `re.findall()` for all matched values;
   - `re.finditer()` when match positions matter;
   - `re.sub()` for pattern-based replacement;
   - `re.split()` for pattern-based splitting.
6. Prefer ordinary string methods when regex would add unnecessary complexity.
7. Treat complex validation patterns as approximations unless the full specification is genuinely required.

Regex is powerful because it lets us describe **patterns in text**, not because every text-cleaning problem needs a regular expression.